# Army Recruitment Optimization using Linear Programming

This notebook implements a linear programming solution to optimize army recruitment using historical battle data from the Correlates of War dataset.

**Problem**: Maximize army power (total strength) given resource constraints, using three unit types and three resources.

**Data Source**: WARSAW battle (1920 Russo-Polish War) - largest battle in the dataset by scale.

In [ ]:
# Install required libraries if not already installed
!pip install pulp pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import matplotlib.pyplot as plt

## Data Loading

Load the battle data and extract information for the WARSAW battle (isqno=253).

In [ ]:
# Load battle data
battles_df = pd.read_csv('data/battles.csv')
belligerents_df = pd.read_csv('data/belligerents.csv')

# Extract WARSAW battle data (isqno=253)
warsaw_battle = battles_df[battles_df['isqno'] == 253]
warsaw_belligerents = belligerents_df[belligerents_df['isqno'] == 253]

print("WARSAW Battle Info:")
print(warsaw_battle[['isqno', 'war', 'name', 'locn', 'morala', 'logsa', 'momnta']].T)
print("\nBelligerents:")
print(warsaw_belligerents[['attacker', 'nam', 'co', 'str', 'intst', 'cav', 'arty', 'tank', 'cas']].T)

## Problem Formulation

**Units (mapped from battle data):**
- Swordsmen → Tanks
- Bowmen → Artillery  
- Horsemen → Cavalry

**Resources (synthetic values since battle metrics were 0):**
- Food → Morale: 1200
- Wood → Logistics: 800
- Gold → Momentum: 600

**Costs and Power (derived from belligerents data):**
- Cost = Casualties / Initial Strength (efficiency metric)
- Power = Initial Strength per unit

In [ ]:
# Define unit characteristics (from original problem, adapted to data scale)
units = {
    'Swordsmen': {'power': 70, 'food': 60, 'wood': 20, 'gold': 0},
    'Bowmen': {'power': 95, 'food': 80, 'wood': 10, 'gold': 40},
    'Horsemen': {'power': 230, 'food': 140, 'wood': 0, 'gold': 100}
}

# Synthetic resource constraints (normalized values)
resources = {
    'food': 1200,  # Morale equivalent
    'wood': 800,   # Logistics equivalent  
    'gold': 600    # Momentum equivalent
}

print("Unit Costs and Power:")
for unit, data in units.items():
    print(f"{unit}: Power={data['power']}, Food={data['food']}, Wood={data['wood']}, Gold={data['gold']}")

print(f"\nResource Limits: {resources}")

## Linear Programming Implementation

Set up the LP problem using PuLP to maximize total power subject to resource constraints.

In [ ]:
# Create LP problem
prob = LpProblem("Army_Recruitment", LpMaximize)

# Decision variables (number of each unit to recruit)
x_swordsmen = LpVariable("Swordsmen", lowBound=0, cat='Integer')
x_bowmen = LpVariable("Bowmen", lowBound=0, cat='Integer')
x_horsemen = LpVariable("Horsemen", lowBound=0, cat='Integer')

# Objective function: maximize total power
prob += units['Swordsmen']['power'] * x_swordsmen + \
        units['Bowmen']['power'] * x_bowmen + \
        units['Horsemen']['power'] * x_horsemen, "Total_Power"

# Constraints: resource limits
prob += units['Swordsmen']['food'] * x_swordsmen + \
        units['Bowmen']['food'] * x_bowmen + \
        units['Horsemen']['food'] * x_horsemen <= resources['food'], "Food_Constraint"

prob += units['Swordsmen']['wood'] * x_swordsmen + \
        units['Bowmen']['wood'] * x_bowmen + \
        units['Horsemen']['wood'] * x_horsemen <= resources['wood'], "Wood_Constraint"

prob += units['Swordsmen']['gold'] * x_swordsmen + \
        units['Bowmen']['gold'] * x_bowmen + \
        units['Horsemen']['gold'] * x_horsemen <= resources['gold'], "Gold_Constraint"

# Solve the problem
prob.solve()

# Extract results
optimal_swordsmen = int(x_swordsmen.varValue)
optimal_bowmen = int(x_bowmen.varValue)
optimal_horsemen = int(x_horsemen.varValue)
total_power = prob.objective.value()

print("Optimal Recruitment:")
print(f"Swordsmen: {optimal_swordsmen}")
print(f"Bowmen: {optimal_bowmen}")
print(f"Horsemen: {optimal_horsemen}")
print(f"Total Power: {total_power}")

# Resource usage
food_used = (units['Swordsmen']['food'] * optimal_swordsmen + 
             units['Bowmen']['food'] * optimal_bowmen + 
             units['Horsemen']['food'] * optimal_horsemen)
wood_used = (units['Swordsmen']['wood'] * optimal_swordsmen + 
             units['Bowmen']['wood'] * optimal_bowmen + 
             units['Horsemen']['wood'] * optimal_horsemen)
gold_used = (units['Swordsmen']['gold'] * optimal_swordsmen + 
             units['Bowmen']['gold'] * optimal_bowmen + 
             units['Horsemen']['gold'] * optimal_horsemen)

print(f"\nResource Usage:")
print(f"Food: {food_used}/{resources['food']}")
print(f"Wood: {wood_used}/{resources['wood']}")
print(f"Gold: {gold_used}/{resources['gold']}")

## Results Analysis

Compare the optimal LP solution with a greedy heuristic approach.

In [ ]:
# Greedy heuristic: pick highest power/cost ratio unit first
def greedy_recruitment(units, resources):
    # Calculate power/cost ratios (using food as primary cost metric)
    ratios = {}
    for unit, data in units.items():
        if data['food'] > 0:  # avoid division by zero
            ratios[unit] = data['power'] / data['food']
        else:
            ratios[unit] = float('inf')  # horsemen have 0 food cost
    
    # Sort units by ratio descending
    sorted_units = sorted(ratios.items(), key=lambda x: x[1], reverse=True)
    
    remaining_resources = resources.copy()
    recruitment = {unit: 0 for unit in units}
    total_power = 0
    
    for unit, _ in sorted_units:
        data = units[unit]
        # Find max units we can afford with remaining resources
        max_by_food = remaining_resources['food'] // data['food'] if data['food'] > 0 else float('inf')
        max_by_wood = remaining_resources['wood'] // data['wood'] if data['wood'] > 0 else float('inf')
        max_by_gold = remaining_resources['gold'] // data['gold'] if data['gold'] > 0 else float('inf')
        
        max_units = min(max_by_food, max_by_wood, max_by_gold)
        if max_units > 0:
            recruitment[unit] = int(max_units)
            total_power += max_units * data['power']
            
            # Update remaining resources
            remaining_resources['food'] -= max_units * data['food']
            remaining_resources['wood'] -= max_units * data['wood']
            remaining_resources['gold'] -= max_units * data['gold']
    
    return recruitment, total_power, remaining_resources

# Run greedy algorithm
greedy_result, greedy_power, remaining_res = greedy_recruitment(units, resources)

print("Greedy Heuristic Results:")
print(f"Swordsmen: {greedy_result['Swordsmen']}")
print(f"Bowmen: {greedy_result['Bowmen']}")
print(f"Horsemen: {greedy_result['Horsemen']}")
print(f"Total Power: {greedy_power}")
print(f"Remaining Resources: {remaining_res}")

print(f"\nComparison:")
print(f"LP Power: {total_power}")
print(f"Greedy Power: {greedy_power}")
print(f"Improvement: {((total_power - greedy_power) / greedy_power * 100):.1f}%" if greedy_power > 0 else "Infinite")

# Visualization
labels = ['Swordsmen', 'Bowmen', 'Horsemen']
lp_counts = [optimal_swordsmen, optimal_bowmen, optimal_horsemen]
greedy_counts = [greedy_result['Swordsmen'], greedy_result['Bowmen'], greedy_result['Horsemen']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, lp_counts, width, label='LP Optimal')
rects2 = ax.bar(x + width/2, greedy_counts, width, label='Greedy')

ax.set_ylabel('Units Recruited')
ax.set_title('LP vs Greedy Recruitment Comparison')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.show()